# Section 06: 从头训练因果语言模型（CLM）核心总结

## 任务定义
**Causal Language Model (CLM)** = 自回归语言模型，每步预测下一个 token。
代表模型：GPT-2、GPT-3、LLaMA 等。

**本节特殊之处：从头训练（from scratch）**，不加载预训练权重，在特定领域代码语料上训练专用代码生成模型。

## 本节任务
在 Python 数据科学代码（使用 pandas/sklearn/matplotlib 的 GitHub 代码）上，
从头训练一个 **GPT-2 架构**的代码生成模型 `codeparrot`。

## CLM vs MLM 对比
```
MLM（BERT类）：                CLM（GPT类）：
[CLS] I [MASK] a dog [SEP]    I love a dog
       ↓                       ↓
预测被遮盖的词                  预测下一个词
双向注意力（看全文）            单向注意力（只看左边）
适合理解任务                   适合生成任务
```

## 完整流程
```
GitHub 代码数据集（过滤含数据科学库的文件）
    ↓ 自定义 Tokenizer
    ↓ tokenize + 切块（return_overflowing_tokens，只保留满 128 的块）
    ↓ DataCollatorForLanguageModeling(mlm=False)  ← CLM 模式
    ↓ GPT2LMHeadModel（从头随机初始化）
    ↓ 自定义 keytoken_weighted_loss
    ↓ 梯度累积 + gradient_accumulation_steps
```

---
## 第一步：数据准备 — 过滤 + Chunk Tokenize

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# 数据过滤逻辑：只保留使用数据科学库的 Python 文件
def any_keyword_in_string(string, keywords):
    return any(keyword in string for keyword in keywords)

filters = ["pandas", "sklearn", "matplotlib", "seaborn"]
# 约 3.26% 的代码文件包含这些关键词

# 使用已过滤好的数据集（避免流式处理的耗时）
ds_train = load_dataset("huggingface-course/codeparrot-ds-train", split="train")
ds_valid = load_dataset("huggingface-course/codeparrot-ds-valid", split="train")
raw_datasets = DatasetDict({"train": ds_train, "valid": ds_valid})

print(f"训练集: {len(raw_datasets['train'])} 个文件")
print(f"验证集: {len(raw_datasets['valid'])} 个文件")

In [ ]:
# 使用专门为代码训练的 BPE Tokenizer（词表针对代码优化）
context_length = 128
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")

# ──────────────────────────────────────────────────────────────
# CLM 的 Tokenize 策略：return_overflowing_tokens
# ──────────────────────────────────────────────────────────────
# 与 MLM 的 group_texts 不同，这里直接在 tokenize 时利用
# return_overflowing_tokens=True 让 tokenizer 自动切块
# return_length=True 返回每块的实际长度，用于过滤不足 128 的块

def tokenize(element):
    outputs = tokenizer(
        element["content"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,  # 超出部分继续切块，而不是直接丢弃
        return_length=True,              # 返回每块实际长度
    )
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        # 只保留满 128 的块，避免 padding（CLM 每步预测下一词，序列越长信息越完整）
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}

tokenized_datasets = raw_datasets.map(
    tokenize, batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print(f"Tokenize 后训练块数: {len(tokenized_datasets['train'])}")
# 606720 个文件 → 1670万个128-token块

---
## 第二步：从头构建 GPT-2 模型

**关键差异**：不使用 `from_pretrained()`，而是用 `AutoConfig` + 模型类直接初始化随机权重。

这意味着所有参数都需要从数据中学习，需要更多数据和更长训练。

In [ ]:
from transformers import AutoConfig, GPT2LMHeadModel

# 自定义 GPT-2 配置
config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),       # 使用代码 tokenizer 的词表大小
    n_ctx=context_length,            # 上下文长度 = 128
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# 随机初始化（不加载预训练权重）
model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 参数量: {model_size/1e6:.1f}M")

# GPT-2 架构：
# Embedding → 12层 Decoder-only Transformer → LM Head
# 每层只有自注意力 + FFN（没有 cross-attention，因为没有 encoder）

---
## 第三步：CLM 的 DataCollator

CLM 的标签是 **input_ids 整体右移一位**：
```
输入:  [I, love, Python, code]
标签:  [love, Python, code, <EOS>]
       ← 每个位置预测下一个 token
```
`DataCollatorForLanguageModeling(mlm=False)` 自动完成这个右移。

In [ ]:
from transformers import DataCollatorForLanguageModeling

# pad_token 不存在时，用 eos_token 代替（GPT-2 通常没有 pad_token）
tokenizer.pad_token = tokenizer.eos_token

# mlm=False → CLM 模式：labels = input_ids 右移，不做随机 mask
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# 演示：查看 CLM collator 的输出
out = data_collator([tokenized_datasets["train"][i] for i in range(5)])
for key in out:
    print(f"{key} shape: {out[key].shape}")
# input_ids shape:     [5, 128]
# attention_mask shape:[5, 128]
# labels shape:        [5, 128]  ← labels == input_ids，loss 在模型内部做右移计算

---
## 第四步：自定义损失函数 — Key Token Weighted Loss

**创新点**：对代码中的「关键词」（plt, pd, fit, predict 等 API 名称）给予更高的权重。
原因：这些 token 在代码中最重要，但数量少，普通 loss 可能忽略它们。

In [ ]:
# 找出关键词的 token id（只保留单 token 的关键词）
keytoken_ids = []
for keyword in ["plt", "pd", "sk", "fit", "predict", " plt", " pd", " fit", " predict"]:
    ids = tokenizer([keyword]).input_ids[0]
    if len(ids) == 1:  # 只取单 token 关键词
        keytoken_ids.append(ids[0])

print(f"关键词 token 数量: {len(keytoken_ids)}")

In [ ]:
import torch
from torch.nn import CrossEntropyLoss

def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
    """
    加权 loss：包含关键词的 batch 样本权重更高。
    
    原理：
    1. 计算每个样本的平均 loss（per_sample loss）
    2. 统计该样本中关键词出现次数，转换为权重
    3. 用权重对 loss 加权平均
    """
    # CLM: token n 预测 token n+1，所以 labels 右移一位
    shift_labels = inputs[..., 1:].contiguous()       # 去掉第一个 token
    shift_logits = logits[..., :-1, :].contiguous()  # 去掉最后一个 logit

    # 计算每个 token 的 loss
    loss_fct = CrossEntropyLoss(reduce=False)
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)

    # 统计每个样本中关键词出现的次数
    # inputs == kt: [batch, seq_len] 的 bool tensor，对所有关键词求和
    weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(axis=[0, 2])
    weights = alpha * (1.0 + weights)  # 至少权重为 1，关键词越多权重越高

    return (loss_per_sample * weights).mean()

---
## 第五步：Accelerate 训练循环（含梯度累积）

**梯度累积（Gradient Accumulation）**：
- 目的：在显存有限时模拟大 batch 训练
- 原理：跑 N 步不更新参数，将梯度累加，第 N 步才更新
- 等效 batch_size = `per_device_batch_size × gradient_accumulation_steps`

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from accelerate import Accelerator
from transformers import get_scheduler

# ── 参数分组：不对 bias 和 LayerNorm 做 weight decay ──────────
weight_decay = 0.1

def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    """将参数分为两组：需要/不需要 weight decay"""
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [
        {"params": params_with_wd,    "weight_decay": weight_decay},
        {"params": params_without_wd, "weight_decay": 0.0},
    ]

# ── 训练配置 ──────────────────────────────────────────────────
gradient_accumulation_steps = 8  # 实际 batch = 32 × 8 = 256
eval_steps = 5_000

# ── 训练循环（关键部分）──────────────────────────────────────
# for epoch in range(num_train_epochs):
#     model.train()
#     for step, batch in enumerate(train_dataloader, start=1):
#
#         # 用自定义 loss，而不是 outputs.loss
#         logits = model(batch["input_ids"]).logits
#         loss = keytoken_weighted_loss(batch["input_ids"], logits, keytoken_ids)
#
#         # 梯度累积：loss 要除以累积步数
#         loss = loss / gradient_accumulation_steps
#         accelerator.backward(loss)
#
#         if step % gradient_accumulation_steps == 0:
#             # 梯度裁剪防止梯度爆炸
#             accelerator.clip_grad_norm_(model.parameters(), 1.0)
#             optimizer.step()
#             lr_scheduler.step()
#             optimizer.zero_grad()

print("训练核心要点:")
print("1. 自定义 keytoken_weighted_loss（不用 outputs.loss）")
print("2. loss /= gradient_accumulation_steps（梯度累积）")
print("3. clip_grad_norm_（梯度裁剪，从头训练容易梯度爆炸）")
print("4. 每 gradient_accumulation_steps 步才 optimizer.step()")

---
## 第六步：代码生成推理

In [ ]:
import torch
from transformers import pipeline

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
pipe = pipeline("text-generation", model="huggingface-course/codeparrot-ds", device=device)

# 演示：代码补全
examples = [
    # 散点图
    """x = np.random.randn(100)\ny = np.random.randn(100)\n# create scatter plot\n""",
    # DataFrame 操作
    """df = pd.DataFrame({'profession': x, 'income':y})\n# calculate the mean income per profession\n""",
    # sklearn 模型
    """from sklearn.ensemble import RandomForestRegressor\n# fit random forest with 300 estimators on X, y:\n""",
]

for txt in examples:
    print(pipe(txt, num_return_sequences=1)[0]["generated_text"])
    print("-" * 40)

# 输出示例：
# plt.scatter(x, y)               ← 正确生成绘图代码
# profession = df.groupby(...).mean()  ← 正确的 pandas 操作
# rf = RandomForestRegressor(n_estimators=300, ...)  ← 正确的 sklearn 调用

---
## 总结

### 核心知识点速查

| 概念 | 说明 |
|------|------|
| CLM 预测目标 | 预测下一个 token，labels = input_ids 右移一位 |
| 从头训练 | `GPT2LMHeadModel(config)` 而非 `from_pretrained()`，所有参数随机初始化 |
| `return_overflowing_tokens=True` | tokenize 时自动切块，配合 `return_length` 过滤不足长度的块 |
| `DataCollatorForLanguageModeling(mlm=False)` | CLM 模式，labels = input_ids，模型内部做右移 |
| 梯度累积 | `loss /= N` → 每 N 步 `optimizer.step()`，等效大 batch |
| 参数分组 | bias/LayerNorm 不做 weight decay，其余参数做 |
| 梯度裁剪 | `clip_grad_norm_(1.0)`，从头训练时尤其重要 |
| keytoken_weighted_loss | 对含关键词的样本加权，提升关键 API 的学习效果 |

### CLM 的 loss 计算
```
输入序列: [t0, t1, t2, t3]
logits:   [l0, l1, l2, l3]   (每个位置预测词表概率)

CLM loss 计算：
shift_logits: [l0, l1, l2]   (去掉最后一个)
shift_labels: [t1, t2, t3]   (去掉第一个)
→ l0 预测 t1，l1 预测 t2，l2 预测 t3
```

### 从头训练 vs 微调
```
从头训练：需要大量数据（本节 ~1.7千万个块）、高学习率(5e-4)、cosine调度
微调：    少量数据即可，低学习率(2e-5)，快速收敛
```